# Prerequisites

## Read All ecommerce.silver tables in dfs

In [0]:
customers_df = spark.read.table("ecommerce.silver.customers")
orders_df = spark.read.table("ecommerce.silver.orders")
order_items_df = spark.read.table("ecommerce.silver.order_items")
products_df = spark.read.table("ecommerce.silver.products")
# payments_df = spark.read.table("ecommerce.silver.payments")
# inventory_df = spark.read.table("ecommerce.silver.inventory")
clickstream_df = spark.read.table("ecommerce.silver.clickstream")

# Calculations

## customer_id , customer_name , first_order_date, last_order_date, total_orders, total_quantity, total_spend,  average_order_value, days_since_last_order

In [0]:
from pyspark.sql.functions import *

customers_orders_items_df = (
    customers_df.alias("c")
    .join(orders_df.alias("o"), "customer_id", "left")
    .join(order_items_df.alias("oi"), "order_id", "left")
)
customer_360_df = (
    customers_orders_items_df.groupBy("customer_id", "first_name", "last_name")
    .agg(
        min(to_date("order_date")).alias("first_order_date"),
        max(to_date("order_date")).alias("last_order_date"),
        countDistinct("order_id").alias("total_orders"),
        sum("quantity").alias("total_quantity"),
        round(sum(col("net_amount")), 2).alias("total_spend"),
    )
    .withColumn(
        "avg_order_val",
        when(
            col("total_orders") > 0, round(col("total_spend") / col("total_orders"), 2)
        ),
    )
    .withColumn(
        "days_since_last_order", date_diff(current_date(), to_date("last_order_date"))
    )
)

customer_360_df.limit(5).display()
customer_360_df.printSchema()

## preferred_category



customer_segment

In [0]:
cust_prod_df = customers_orders_items_df.alias("coi").join(
    products_df.alias("p"), "product_id", "left"
)
# cust_prod_df.limit(5).display()
cust_prod_df = cust_prod_df.groupBy("customer_id", "category").count()
from pyspark.sql.window import Window

window_spec = Window.partitionBy("customer_id").orderBy(desc(col("count")))
cust_prod_df = (
    cust_prod_df.withColumn("rank", row_number().over(window_spec))
    .filter(col("rank") == 1)
    .select(col("customer_id"), col("category").alias("preferred_category"))
)
cust_prod_df.limit(5).display()

## preferred_device

In [0]:
cust_click_df = (
    customers_df.alias("c")
    .join(clickstream_df.alias("cs"), "customer_id", "left")
    .groupBy("customer_id", "device_type")
    .count()
)

window_spec = Window.partitionBy("customer_id").orderBy(desc(col("count")))
cust_click_df = (
    cust_click_df.withColumn("rank", row_number().over(window_spec))
    .filter(col("rank") == 1)
    .select(col("customer_id"), col("device_type").alias("preferred_device_type"))
)
cust_click_df.limit(5).display()

## customer_segment

In [0]:
customer_360_df.printSchema()

In [0]:

customer_360_df = (
    customer_360_df.join(cust_prod_df, "customer_id", "left")
    .join(cust_click_df, "customer_id", "left")
    .withColumn("customer_name", concat_ws(" ","first_name", "last_name"))
    .withColumn(
        "customer_segment",
        when(col("total_spend") >= 5000, "VIP")
        .when(col("total_spend") >= 2000, "HIGH_VALUE")
        .when(col("total_orders") >= 10, "LOYAL")
        .when(col("days_since_last_order") <= 30, "ACTIVE")
        .otherwise("INACTIVE"),
    )
)

customer_360_df.limit(10).display()

# Create gold.customer_360 Delta Table

In [0]:
customer_360_df.write.format("delta").mode("overwrite").saveAsTable("ecommerce.gold.customer_360")
spark.read.table("ecommerce.gold.customer_360").display()